In [1]:
import numpy as np
import pandas as pd

In [2]:
temp_df = pd.read_csv('IMDBDataset.csv')

In [3]:
df = temp_df.iloc[:10000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df = df.drop_duplicates()

In [6]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [7]:
df['review'] = df['review'].str.lower()

In [8]:
import nltk
from nltk.corpus import stopwords

# Download stopwords corpus if not already downloaded
nltk.download('stopwords')

# Define the stop words list/set
sw_set = set(stopwords.words('english'))

# Run the stop words removal
df['review'] = df['review'].apply(
    lambda x: " ".join([word for word in str(x).split() if word not in sw_set])
)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jaswanth/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df['review']

0       one reviewers mentioned watching 1 oz episode ...
1       wonderful little production. <br /><br />the f...
2       thought wonderful way spend time hot summer we...
3       basically there's family little boy (jake) thi...
4       petter mattei's "love time money" visually stu...
                              ...                        
9995    fun, entertaining movie wwii german spy (julie...
9996    give break. anyone say "good hockey movie"? kn...
9997    movie bad movie. watching endless series bad h...
9998    movie probably made entertain middle school, e...
9999    smashing film film-making. shows intense stran...
Name: review, Length: 9983, dtype: str

In [10]:
import gensim

In [11]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [12]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [13]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [14]:
model.build_vocab(story)

In [15]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5909241, 6399515)

In [16]:
len(model.wv.index_to_key)

31846

In [17]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)


In [19]:
document_vector(df['review'].values[0])

array([-0.20273964,  0.34332985,  0.16127804,  0.18450624,  0.13895607,
       -0.55978537,  0.02173419,  0.80773425, -0.33076146, -0.04671543,
       -0.12978457, -0.52267146, -0.04553226,  0.14640543,  0.17379375,
       -0.23984979,  0.28154048, -0.3232392 , -0.04980549, -0.94016063,
        0.06949799,  0.23874097,  0.25082085, -0.02712669, -0.0252022 ,
       -0.08243933, -0.15966353, -0.21902803, -0.44422516, -0.12156768,
        0.39922398, -0.02327557,  0.10650249, -0.34756076,  0.02204228,
        0.5622776 ,  0.06241857, -0.4331054 , -0.3673806 , -0.48038068,
        0.15455018, -0.45309293, -0.17081708, -0.03697067,  0.3369682 ,
       -0.27156574, -0.33090022,  0.05432139,  0.27646   ,  0.31005716,
        0.11850267, -0.39035133, -0.30171645,  0.14286394, -0.19763416,
        0.1798453 ,  0.22431266, -0.12197572, -0.13279298,  0.04716612,
       -0.11197881,  0.12735273, -0.05114539, -0.04384967, -0.48223308,
        0.3543086 ,  0.18558705,  0.31748524, -0.27872518,  0.48

In [20]:
from tqdm import tqdm

In [21]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))


100%|██████████| 9983/9983 [03:49<00:00, 43.52it/s]


In [22]:
X = np.array(X)

In [23]:
X[0]

array([-0.20273964,  0.34332985,  0.16127804,  0.18450624,  0.13895607,
       -0.55978537,  0.02173419,  0.80773425, -0.33076146, -0.04671543,
       -0.12978457, -0.52267146, -0.04553226,  0.14640543,  0.17379375,
       -0.23984979,  0.28154048, -0.3232392 , -0.04980549, -0.94016063,
        0.06949799,  0.23874097,  0.25082085, -0.02712669, -0.0252022 ,
       -0.08243933, -0.15966353, -0.21902803, -0.44422516, -0.12156768,
        0.39922398, -0.02327557,  0.10650249, -0.34756076,  0.02204228,
        0.5622776 ,  0.06241857, -0.4331054 , -0.3673806 , -0.48038068,
        0.15455018, -0.45309293, -0.17081708, -0.03697067,  0.3369682 ,
       -0.27156574, -0.33090022,  0.05432139,  0.27646   ,  0.31005716,
        0.11850267, -0.39035133, -0.30171645,  0.14286394, -0.19763416,
        0.1798453 ,  0.22431266, -0.12197572, -0.13279298,  0.04716612,
       -0.11197881,  0.12735273, -0.05114539, -0.04384967, -0.48223308,
        0.3543086 ,  0.18558705,  0.31748524, -0.27872518,  0.48

In [24]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [25]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [26]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [28]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7741612418627942